All verified paired volumes from Samples04–17. Samples18–20 remain held out, using the historical 36-volume benchmark. Three passes through the full training cohort. The benchmark uses generated reference masks.

In [ ]:

import sys, subprocess, json, re, hashlib
from pathlib import Path
subprocess.run([sys.executable,'-m','pip','install','-q','nibabel','beautifulsoup4'],check=True)
import nibabel, requests
from bs4 import BeautifulSoup
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from scipy import ndimage as ndi
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from skimage.feature import peak_local_max
def normalize(a):
    lo, hi = np.percentile(a, (1, 99.8))
    return np.clip((a.astype(np.float32)-lo)/max(1,hi-lo),0,1)
def block(a,b):
    return nn.Sequential(nn.Conv3d(a,b,3,padding=1),nn.GroupNorm(4,b),nn.SiLU(),
                         nn.Conv3d(b,b,3,padding=1),nn.GroupNorm(4,b),nn.SiLU())
class Detector(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc, self.deep, self.dec, self.head = block(1,8), block(8,16), block(24,8), nn.Conv3d(8,1,1)
    def forward(self,x):
        a=self.enc(x)
        b=self.deep(F.avg_pool3d(a,(1,2,2)))
        return self.head(self.dec(torch.cat([a,F.interpolate(b,size=a.shape[2:],mode='trilinear',align_corners=False)],1)))


assert torch.cuda.is_available(), 'Kaggle GPU required'
torch.manual_seed(42)
rng=np.random.default_rng(42)
out=Path('/kaggle/working')
cache=Path('/tmp/cshaper')
cache.mkdir(exist_ok=True)
train=[f'Sample{i:02d}' for i in range(4,18)]
test=['Sample18','Sample19','Sample20']

def listing(folder):
    response=requests.get('https://drive.google.com/embeddedfolderview',params={'id':folder},timeout=40)
    response.raise_for_status()
    entries={}
    for anchor in BeautifulSoup(response.text,'html.parser').select('a'):
        title=anchor.select_one('.flip-entry-title')
        match=re.search(r'/(?:folders|d)/([\w-]+)',anchor.get('href',''))
        if title and match:
            entries[title.get_text(strip=True)]=match[1]
    return entries

def frames(entries, sample, suffix):
    pattern=rf'{sample}_(\d+)_{suffix}\.nii\.gz'
    return {int(m[1]): key for name,key in entries.items() if (m:=re.fullmatch(pattern,name))}

def fetch(key):
    path=cache/f'{key}.nii.gz'
    if not path.exists():
        from requests.adapters import HTTPAdapter
        from urllib3.util.retry import Retry
        with requests.Session() as session:
            session.mount('https://', HTTPAdapter(max_retries=Retry(total=3, backoff_factor=2,
                status_forcelist=(429, 500, 502, 503, 504))))
            with session.get('https://drive.usercontent.google.com/download',
                             params=dict(id=key, export='download', confirm='t'),
                             stream=True, timeout=(20, 120)) as response:
                response.raise_for_status()
                chunks = response.iter_content(1024 * 1024)
                head = next(chunks, b'')
                if not head.startswith(bytes((31, 139))):
                    raise ValueError(f'Drive did not serve a gzip file for {key}')
                staging = path.with_suffix('.part')
                with staging.open('wb') as output:
                    output.write(head)
                    for chunk in chunks:
                        output.write(chunk)
                staging.replace(path)
    return nibabel.load(path), hashlib.sha256(path.read_bytes()).hexdigest()

folders=listing('151pBI7d_JfkOz4eQI4R4F2u_cKtlftBi')
manifest={}
for sample in train+test:
    children=listing(folders[sample])
    images=frames(listing(children['RawNuc']),sample,'rawNuc')
    masks=frames(listing(children['SegNuc']),sample,'segNuc')
    paired=sorted(images.keys() & masks.keys())
    sample_count = len(paired) if sample in train else 12
    assert len(paired) >= sample_count, (sample,len(paired))
    chosen=[paired[i] for i in np.linspace(0,len(paired)-1,sample_count,dtype=int)]
    manifest[sample]=[dict(t=t,image=images[t],mask=masks[t]) for t in chosen]
    print(sample,len(paired),flush=True)
(out/'split.json').write_text(json.dumps(dict(train=train,test=test,frames=manifest),indent=2))

def load(row):
    a, row['image_sha256']=fetch(row['image'])
    b, row['mask_sha256']=fetch(row['mask'])
    assert a.shape==b.shape and len(a.shape)==3 and np.allclose(a.affine,b.affine,atol=1e-4), 'Image and mask grids differ'
    spacing=np.asarray(a.header.get_zooms()[::-1])
    image=normalize(np.asarray(a.dataobj).transpose(2,1,0))
    labels=np.asarray(b.dataobj).transpose(2,1,0)
    ids=np.unique(labels)
    points=np.asarray(ndi.center_of_mass(labels>0,labels,ids[ids>0])).reshape(-1,3)
    assert len(points), 'Empty nucleus mask'
    return image,points,spacing

net=Detector().cuda()
opt=torch.optim.AdamW(net.parameters(),lr=0.001)
updates=0
training_rows=[row for sample in train for row in manifest[sample]]
for epoch in range(1,4):
    for index in rng.permutation(len(training_rows)):
        image,points,_=load(training_rows[index])
        points=points.round().astype(int)
        size=np.minimum(image.shape,(16,64,64))
        for _ in range(4):
            start=np.clip(points[rng.integers(len(points))]-size//2+rng.integers(-6,7,3),0,np.asarray(image.shape)-size)
            slices=tuple(slice(int(a),int(a+b)) for a,b in zip(start,size))
            local=points-start
            local=local[((local>=0)&(local<size)).all(axis=1)]
            target=np.zeros(tuple(size),dtype=np.float32)
            target[tuple(local.T)]=1
            target=ndi.maximum_filter(target,size=(1,3,3))
            x=torch.as_tensor(image[slices].copy(),dtype=torch.float32,device='cuda')[None,None]
            y=torch.as_tensor(target,device='cuda')[None,None]
            loss=F.binary_cross_entropy_with_logits(net(x),y,pos_weight=x.new_tensor(30.))
            assert torch.isfinite(loss)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            updates+=1
    print(dict(epoch=epoch,updates=updates),flush=True)
    torch.save(dict(model=net.state_dict(),epoch=epoch,updates=updates),out/'detector.pt')

net.eval()
results={}
with torch.inference_mode():
    for sample in test:
        rows=[]
        for row in manifest[sample]:
            image,points,spacing=load(row)
            scores=np.empty_like(image,dtype=np.float32)
            for start in range(0,len(image),8):
                lo,hi=max(0,start-4),min(len(image),start+12)
                end=min(len(image),start+8)
                x=torch.as_tensor(image[lo:hi],dtype=torch.float32,device='cuda')[None,None]
                scores[start:end]=net(x).sigmoid()[0,0,start-lo:end-lo].cpu().numpy()
            pred=peak_local_max(scores,threshold_abs=0.5,min_distance=2,exclude_border=False)
            d=cdist(points,pred)
            i,j=linear_sum_assignment(np.where(d<=3,d,(min(d.shape)+1)*4))
            rows.append(dict(t=row['t'],matched=int((d[i,j]<=3).sum()),annotated=len(points),predicted=len(pred)))
        results[sample]=rows
(out/'summary.json').write_text(json.dumps(dict(train=train,test=test,frames=manifest,updates=updates,match_gate_voxels=3,
    label_type='published generated nucleus masks',results=results),indent=2))
